# Carregamento dos documentos

- carregamento da anamnses
- Aplicada mascara sobre os dados pessoais como cpf, rg, data nascimento, etc.

In [1]:
from langchain_community.document_loaders import TextLoader

from simple_rag.utils.data_masking import mask_pii

docs = None

anamnese1 = "./data/anamnese/anamnese1.txt"

loader = TextLoader(anamnese1)

docs = loader.load()

for doc in docs:
    doc.page_content = mask_pii(doc.page_content, pii_types=['all'])

print(len(docs))

/home/flavio/github/processamento-linguagem-natural-puc-minas/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1


Visualização da máscara aplicada ao documento

In [2]:
print(f"{docs[0].page_content[:500]}\n")
print(docs[0].metadata)

Anamnese Médica — Caso Clínico João Gabriel.

## Identificação do paciente
- Nome: J*** G******
- Idade: 72 anos
- RG: 12.***.***-9 SSP/SP
- CPF: 123.***.***-00
- Data de Nascimento: **/**/****
- Prontuário: ****532
- Telefone Residencial: (**) ****-4487
- Celular: (**) *****-3321
- E-mail: joao************@email.com


## Queixa Principal (QP)
- Urina com sangue há 8 dias (hematúria)

## História da Doença Atual (HDA)
- Paciente com histórico de litíase vesical, submetido à cistolitotomia há 9 m

{'source': './data/anamnese/anamnese1.txt'}


## Split & Chucnks

Separação dos documentos em chunks ou pedaços menores, para melhorar a performance de busca.

A vetorização de texto muito grandes acarreta em perda de precisão.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(f"{len(all_splits)} chunks")

6 chunks


## Visualização dos vetores de embedding

Utilizado modelo llama3 para embedding foram montados 2 vetores a partir dos primeiros chunks para visualização e demonstração.

Os vetores possuem 4096 dimensões (para o modelo de embedding llama3).

In [4]:
from langchain_ollama import OllamaEmbeddings

llama = OllamaEmbeddings(
    model="llama3",
)

vector_1 = llama.embed_query(all_splits[0].page_content)
vector_2 = llama.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])


Generated vectors of length 4096

[-0.016785007, -0.021554101, -0.0092353, 0.017139941, 0.006986494, -0.012714945, -0.031726208, 0.011648087, -0.023892332, 0.005708538]


Independente do tamanho da string a dimensão do vetor é a mesma para o mesmo modelo de embedding.

In [5]:
small_string_vector = llama.embed_query("pequena string")

print(f"Generated vectors of length doc1 {len(vector_1)}\n")
print(f"Generated vectors of length doc2 {len(small_string_vector)}\n")


Generated vectors of length doc1 4096

Generated vectors of length doc2 4096



## Instanciando o vector store

instanciação do vectorstore e criação da collection.

In [6]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="my_collection",
    embedding_function=llama,
    persist_directory="./chromadb_notebook",
)


## Carregando os documentos no vectorstore
Indexação e visualização dos IDS dos chunks dos documentos no vectorstore

In [7]:
ids = vector_store.add_documents(documents=all_splits)
print(ids)

['19de835d-d077-4a82-8cbc-365df770d120', '75359559-19ac-4312-ac1d-1be100474a20', '1d35c51a-e7d2-4b2a-8e08-cf32423ab77b', '88851c51-0d86-4696-85f9-3ffb74c8383f', 'dac2f98f-654b-416a-9be3-68f52c886c26', '28a1b0f4-818f-4380-96b9-93a36f15a34c']


## query

Fazendo busca por similaridade no vectorestore.

Por baixo dos panos é utilizado similaridade por cosseno, mas é possível a utilização de outros métdos de comparação de vetores.

In [8]:
retrivier = vector_store.as_retriever(
    search_type="similarity", search_kwargs={"k": 3}
)

query = "Qual é o História da Doença Atual do paciente?"
documents = retrivier.invoke(query)

print(f"K de documentos retornados: {len(documents)}\n")
print(documents[0].page_content)  # print apenas do primeiro resultado

K de documentos retornados: 3

## Queixa Principal (QP)
- Urina com sangue há 8 dias (hematúria)

## História da Doença Atual (HDA)
- Paciente com histórico de litíase vesical, submetido à cistolitotomia há 9 meses.
- Evoluiu no pós-operatório com incontinência urinária e episódios mensais recorrentes de hematúria.
- Há 8 dias, apresentou novo episódio de hematúria, associado a oligúria e disúria, sem febre.
- Em seguimento com urologista, foi encaminhado ao HUMAP para formolização da bexiga.
- No dia seguinte ao procedimento, evoluiu com ascite, sendo necessária cirurgia de urgência (ureterostomia e cateterismo vesical).
- No momento, encontra-se estável, sem dor, aguardando retorno do trânsito intestinal, em uso de dexametasona e paracetamol.


In [9]:
n = 0
for doc in documents:
    print("=" * 50)
    print(f"Documento: {n}\n")
    print(f"Metadata: {doc.metadata}\nContent: {doc.page_content}\n")
    print("=" * 50)
    n += 1

Documento: 0

Metadata: {'start_index': 321, 'source': './data/anamnese/anamnese1.txt'}
Content: ## Queixa Principal (QP)
- Urina com sangue há 8 dias (hematúria)

## História da Doença Atual (HDA)
- Paciente com histórico de litíase vesical, submetido à cistolitotomia há 9 meses.
- Evoluiu no pós-operatório com incontinência urinária e episódios mensais recorrentes de hematúria.
- Há 8 dias, apresentou novo episódio de hematúria, associado a oligúria e disúria, sem febre.
- Em seguimento com urologista, foi encaminhado ao HUMAP para formolização da bexiga.
- No dia seguinte ao procedimento, evoluiu com ascite, sendo necessária cirurgia de urgência (ureterostomia e cateterismo vesical).
- No momento, encontra-se estável, sem dor, aguardando retorno do trânsito intestinal, em uso de dexametasona e paracetamol.

Documento: 1

Metadata: {'source': './data/anamnese/anamnese1.txt', 'start_index': 3222}
Content: ### Sistema Cardíaco
- Sem turgência jugular ou frêmitos
- Bulhas normofonéticas

In [10]:
response = vector_store.similarity_search_with_score(
    query="Qual é a identificação completa do paciente ?",
    k=3
)

for k, v in response:
    print("=" * 50)
    print(f"Conteúdo:\n{k}\n")
    print(f"Score: {v}\n")
    print("=" * 50)



Conteúdo:
page_content='## Queixa Principal (QP)
- Urina com sangue há 8 dias (hematúria)

## História da Doença Atual (HDA)
- Paciente com histórico de litíase vesical, submetido à cistolitotomia há 9 meses.
- Evoluiu no pós-operatório com incontinência urinária e episódios mensais recorrentes de hematúria.
- Há 8 dias, apresentou novo episódio de hematúria, associado a oligúria e disúria, sem febre.
- Em seguimento com urologista, foi encaminhado ao HUMAP para formolização da bexiga.
- No dia seguinte ao procedimento, evoluiu com ascite, sendo necessária cirurgia de urgência (ureterostomia e cateterismo vesical).
- No momento, encontra-se estável, sem dor, aguardando retorno do trânsito intestinal, em uso de dexametasona e paracetamol.' metadata={'start_index': 321, 'source': './data/anamnese/anamnese1.txt'}

Score: 0.7098280191421509

Conteúdo:
page_content='### Sistema Cardíaco
- Sem turgência jugular ou frêmitos
- Bulhas normofonéticas, ritmo regular, dois tempos, sem sopros

### 